# EXP_020E — Image Ablation: SigLIP + XLM-R + Concat + MSE
**Phase 2 | Image Backbone Ablation**
Research question: Does SigLIP provide better visual features than ConvNeXt for noisy restaurant review images?
- Image model: `google/siglip-base-patch16-256` | Text: `xlm-roberta-base` (from EXP_010)
- Fusion: Concatenation + MLP | Loss: MSE | Seed: 42 | AMP: enabled
> ⚠️ Prerequisite: EXP_010 must be completed and saved to Drive.

### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### STEP 2: Clone source code and install dependencies

In [ ]:
!git clone https://github.com/lechihoang/SE365.git
%cd SE365
!pip install -r requirements.txt -q

### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!gdown --id 11WoeUn2visKtGN5oOX9c2I6Grz3P88vD -O data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

### STEP 4: Configure paths

In [ ]:
import os
DRIVE_ROOT = '/content/drive/MyDrive/SE365'  # ✏️ Change if needed
EXP_ID = 'EXP_020E_siglip_xlmr_concat_mse'
DRIVE_EXP_PATH = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
os.makedirs(DRIVE_EXP_PATH, exist_ok=True)
print(f'Artifacts will be saved to: {DRIVE_EXP_PATH}')

### STEP 5: Load text weights from EXP_010 (SigLIP uses HuggingFace pretrained weights)

In [ ]:
import os, shutil
os.makedirs('./checkpoints', exist_ok=True)
shutil.copy(f'{DRIVE_ROOT}/experiments/EXP_010_text_only_xlmr_mse/best_model_train_text.pth', './checkpoints/best_model_train_text.pth')
print('Loaded text weights from EXP_010')
if os.path.exists('./checkpoints/best_model_train_image.pth'):
    os.remove('./checkpoints/best_model_train_image.pth')
    print('Removed old image checkpoint — SigLIP loads HuggingFace pretrained weights automatically')

### STEP 6: Train

In [ ]:
!python main.py \
  --mode train_fusion \
  --text_model_name xlm-roberta-base \
  --image_model_name google/siglip-base-patch16-256 \
  --epochs 15 \
  --batch_size 8 \
  --lr 1e-5 \
  --grad_accum_steps 4 \
  --patience 5 \
  --loss_fn mse \
  --unfreeze_text_layers 1 \
  --unfreeze_image_layers 1 \
  --seed 42 \
  --use_amp \
  --exp_id EXP_020E_siglip_xlmr_concat_mse \
  --exp_dir ./experiments

### STEP 7: Save to Drive + print metrics

In [ ]:
import shutil, json
src = f'./experiments/{EXP_ID}'
shutil.copytree(src, DRIVE_EXP_PATH, dirs_exist_ok=True)
print(f'Saved to {DRIVE_EXP_PATH}')
with open(f'{src}/metrics.json') as f:
    m = json.load(f)
print(f"\n=== EXP_020E Results ===")
print(f"Mean MAE: {m['mean_mae']:.4f} | Overall MAE: {m['overall_mae']:.4f}")
for k in ['food','price','atmos','service']: print(f"  {k}: {m[f'mae_{k}']:.4f}")